# Problem Book P1, Chapter 2: Roots of Unity Everywhere (Python companion)

Part A checks every number of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `scipy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import itertools, math, cmath, random
from fractions import Fraction as Fr
from collections import Counter
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
random.seed(2)
x, q = sp.symbols('x q')

## Part A: checks

**Problem 2.1 (subset sums).** Brute force against the formula for $N(n,k)$, and the two special cases.

In [ ]:
def N_brute(n, k):
    return sum(1 for S in itertools.combinations(range(n), k) if sum(S) % n == 0)
def N_formula(n, k):
    g = math.gcd(n, k)
    return sum((-1)**(k + k // d) * sp.totient(d) * math.comb(n // d, k // d) for d in sp.divisors(g)) // n
print(all(N_brute(n, k) == N_formula(n, k) for n in range(1, 14) for k in range(n + 1)))
for p in [3, 5, 7]:
    a = sum(N_brute(p, k) for k in range(p + 1))
    b = sum(1 for S in itertools.combinations(range(1, 2 * p + 1), p) if sum(S) % p == 0)
    print(p, a, (2**p - 2) // p + 2, b, (math.comb(2 * p, p) - 2) // p + 2)

**Problem 2.2 (dice modulo $m$).** Exact laws of $S_n$ modulo $5$, $6$, $7$ with fractions.

In [ ]:
def p_div(n, m):
    d = {0: Fr(1)}
    for _ in range(n):
        e = Counter()
        for s, pr in d.items():
            for f in range(1, 7):
                e[(s + f) % m] += pr / 6
        d = e
    return d[0]
for n in range(1, 7):
    print(n, p_div(n, 7) == Fr(1, 7) * (1 + 6 * Fr(-1, 6)**n), p_div(n, 6),
          p_div(n, 5), p_div(n, 5) == Fr(1, 5) + Fr(5 * (n % 5 == 0) - 1, 5 * 6**n))

**Problem 2.3 (Ramanujan sums).** The divisor formula, Hölder's formula, the orthogonality, and Ramanujan's expansion
$\sigma(n)/n=\frac{\pi^2}6\sum_q c_q(n)/q^2$ (a slowly converging series).

In [ ]:
def c_direct(qq, n):
    return sum(cmath.exp(2j * math.pi * a * n / qq) for a in range(1, qq + 1) if math.gcd(a, qq) == 1).real
def c(qq, n):
    return sum(sp.mobius(qq // d) * d for d in sp.divisors(math.gcd(qq, n)))
def holder(qq, n):
    g = math.gcd(qq, n)
    return sp.mobius(qq // g) * sp.totient(qq) // sp.totient(qq // g)
print(all(abs(c_direct(qq, n) - c(qq, n)) < 1e-8 and c(qq, n) == holder(qq, n) for qq in range(1, 30) for n in range(40)))
N = 36
print(all(sum(c(a, n) * c(b, n) for n in range(1, N + 1)) == (N * sp.totient(a) if a == b else 0)
          for a in sp.divisors(N) for b in sp.divisors(N)))
for n in [1, 6, 12]:
    print(n, sp.divisor_sigma(n) / n, math.pi**2 / 6 * sum(c(qq, n) / qq**2 for qq in range(1, 4000)))

**Problem 2.4 (bricks).** For which $a,b\le12$ can an $a\times b$ board be tiled by $1\times3$ bricks? A tiny exact-cover
search agrees with "$3\mid a$ or $3\mid b$".

In [ ]:
def tileable(a, b, k):
    grid = [[False] * b for _ in range(a)]
    def first_empty():
        for i in range(a):
            for j in range(b):
                if not grid[i][j]:
                    return i, j
        return None
    def solve():
        cell = first_empty()
        if cell is None:
            return True
        i, j = cell
        for di, dj in ((0, 1), (1, 0)):
            cells = [(i + di * t, j + dj * t) for t in range(k)]
            if all(r < a and s < b and not grid[r][s] for r, s in cells):
                for r, s in cells: grid[r][s] = True
                if solve(): return True
                for r, s in cells: grid[r][s] = False
        return False
    return solve()
print(all(tileable(a, b, 3) == (a % 3 == 0 or b % 3 == 0) for a in range(1, 9) for b in range(1, 9)))
print("10x10 with 1x4:", tileable(10, 10, 4))

**Problem 2.5 (Prouhet--Thue--Morse).** The moments $\sum t_nn^k$ and the split of $\{0,\dots,15\}$.

In [ ]:
t = lambda n: (-1)**bin(n).count('1')
for N in range(1, 8):
    print(N, [sum(t(n) * n**k for n in range(2**N)) for k in range(N + 1)],
          (-1)**N * math.factorial(N) * 2**(N * (N - 1) // 2))
A = [n for n in range(16) if t(n) == 1]; B = [n for n in range(16) if t(n) == -1]
print(A, B, [(sum(a**k for a in A), sum(b**k for b in B)) for k in range(5)])

**Problem 2.6 (Lights Out on a ring).** Kernel dimension of the pressing matrix over $\mathbb F_2$, and the parity test.

In [ ]:
def rank_f2(M):
    M = [row[:] for row in M]; r = 0
    for col in range(len(M[0])):
        piv = next((i for i in range(r, len(M)) if M[i][col]), None)
        if piv is None: continue
        M[r], M[piv] = M[piv], M[r]
        for i in range(len(M)):
            if i != r and M[i][col]:
                M[i] = [u ^ v for u, v in zip(M[i], M[r])]
        r += 1
    return r
def ring(n):
    return [[1 if (j - i) % n in (0, 1, n - 1) else 0 for j in range(n)] for i in range(n)]
print([(n, n - rank_f2(ring(n))) for n in range(3, 16)])
n = 9
image = set()
for u in itertools.product([0, 1], repeat=n):
    image.add(tuple(sum(u[j] for j in range(n) if (i - j) % n in (0, 1, n - 1)) % 2 for i in range(n)))
test = lambda c: len({sum(c[i] for i in range(n) if i % 3 == r) % 2 for r in range(3)}) == 1
print(len(image), 2**n // 4, all(test(c) == (c in image) for c in itertools.product([0, 1], repeat=n)))

**Problem 2.7 (cyclic sieving).** The $q$-binomial at roots of unity against the number of subsets fixed by rotations.

In [ ]:
def qbinom(n, k):
    num = sp.prod([1 - q**(n - k + i) for i in range(1, k + 1)])
    den = sp.prod([1 - q**i for i in range(1, k + 1)])
    return sp.Poly(sp.cancel(num / den), q)
ok = True
for n in range(1, 11):
    for k in range(n + 1):
        P = qbinom(n, k)
        for j in range(n):
            z = cmath.exp(2j * math.pi * j / n)
            val = sum(complex(cf) * z**e for (e,), cf in P.terms())
            fixed = sum(1 for S in itertools.combinations(range(n), k) if {(s + j) % n for s in S} == set(S))
            ok &= abs(val - fixed) < 1e-7
print(ok)

**Problem 2.8 ($\Phi_n(\pm1)$).** Both values for $n<300$, and the unit $1-\zeta$ for $n=6$ and $n=15$ (its inverse is the
product of the other factors).

In [ ]:
def Lam(n):
    f = sp.factorint(n)
    return list(f)[0] if len(f) == 1 else 1
def at_minus1(n):
    if n & (n - 1) == 0: return 2
    m = n // 2
    if n % 2 == 0 and m % 2 == 1 and len(sp.factorint(m)) == 1: return list(sp.factorint(m))[0]
    return 1
print(all(sp.cyclotomic_poly(n, 1) == Lam(n) for n in range(2, 300)),
      all(sp.cyclotomic_poly(n, -1) == at_minus1(n) for n in range(3, 300)))
for n in [6, 15]:
    z = cmath.exp(2j * math.pi / n)
    print(n, np.prod([1 - z**a for a in range(1, n) if math.gcd(a, n) == 1]))

**Problem 2.9 ($\Phi_{pq}$ and $\Phi_{105}$).** Coefficients in $\{-1,0,1\}$, alternating, for all $p<q<60$; the formula
$\mathbf 1_S(m)-\mathbf 1_S(m-1)$; and the first $-2$.

In [ ]:
ok = True
for p in sp.primerange(2, 30):
    for qq in sp.primerange(p + 1, 60):
        co = sp.Poly(sp.cyclotomic_poly(p * qq, x), x).all_coeffs()[::-1]
        S = {i * p + j * qq for i in range(qq) for j in range(p)}
        ok &= all(co[m] == (m in S) - (m - 1 in S) for m in range(len(co)))
        nz = [v for v in co if v]
        ok &= all(nz[i] == -nz[i + 1] for i in range(len(nz) - 1))
print(ok)
co = sp.Poly(sp.cyclotomic_poly(105, x), x).all_coeffs()[::-1]
print([(k, v) for k, v in enumerate(co) if abs(v) > 1])

**Problem 2.10 (vanishing sums).** Count the non-empty sets of $n$-th roots of unity with sum $0$, and test (c) for
$n=8$ and $n=9$.

In [ ]:
def vanishing(n):
    w = [cmath.exp(2j * math.pi * k / n) for k in range(n)]
    return [S for r in range(1, n + 1) for S in itertools.combinations(range(n), r) if abs(sum(w[k] for k in S)) < 1e-9]
print({n: len(vanishing(n)) for n in [5, 6, 7, 8, 9, 10, 12]})
for n, p in [(8, 2), (9, 3)]:
    N = n // p
    print(n, all(all((r in S) == (r % N in {s % N for s in S}) for r in range(n)) for S in vanishing(n)))

**Problem 2.11 (Kronecker).** All monic integer polynomials of degree $\le4$ with $P(0)\ne0$ and roots in the closed unit
disc: every one is a product of cyclotomic polynomials. And $\frac{3+4i}5$ is not a root of unity.

In [ ]:
cyclo = {sp.Poly(sp.cyclotomic_poly(n, x), x) for n in range(1, 40) if sp.totient(n) <= 4}
count = 0
for d in range(1, 5):
    for co in itertools.product(*[range(-math.comb(d, j), math.comb(d, j) + 1) for j in range(1, d + 1)]):
        if co[-1] == 0: continue
        P = sp.Poly([1, *co], x)
        if max(abs(r) for r in np.roots([1, *co])) <= 1 + 1e-3:
            count += 1
            assert all(sp.Poly(f, x) in cyclo for f, _ in sp.factor_list(P.as_expr())[1])
print(count, "polynomials, all products of cyclotomic ones")
z = (3 + 4j) / 5
print(min(abs(z**k - 1) for k in range(1, 5000)))

**Problem 2.12 (primitive roots).** Sums of $k$-th powers of primitive roots against $c_{p-1}(k)$ modulo $p$.

In [ ]:
ok = True
for p in sp.primerange(3, 80):
    prim = [g for g in range(1, p) if sp.n_order(g, p) == p - 1]
    ok &= all(sum(pow(g, k, p) for g in prim) % p == c(p - 1, k) % p for k in range(0, 2 * p))
    ok &= p == 3 or math.prod(prim) % p == 1
print(ok)

**Problem 2.13 ($\Phi_n$ modulo $p$).** Number and degrees of the factors, $X^4+1$ modulo small primes, and (c).

In [ ]:
ok = True
for n in range(2, 40):
    for p in sp.primerange(2, 30):
        fl = sp.factor_list(sp.cyclotomic_poly(n, x), modulus=p)[1]
        if n % p:
            f = sp.n_order(p, n)
            ok &= len(fl) == sp.totient(n) // f and all(sp.degree(g, x) == f and e == 1 for g, e in fl)
        else:
            a = sp.multiplicity(p, n); m = n // p**a
            lhs = sp.Poly(sp.cyclotomic_poly(n, x), x, modulus=p)
            rhs = sp.Poly(sp.cyclotomic_poly(m, x), x, modulus=p)**int(sp.totient(p**a))
            ok &= lhs == rhs
print(ok)
print([(p, sp.factor_list(x**4 + 1, modulus=p)[1]) for p in [2, 3, 5, 7, 17]])

**Problem 2.14 (rational tangents, Niven).** Rational multiples of $\pi$ with denominator up to $120$: the values of
$\tan$ and of $\cos$ that are fractions with denominator at most $1000$ (to $12$ digits).

In [ ]:
tans, coss = set(), set()
for den in range(1, 121):
    for num in range(0, 2 * den):
        th = math.pi * num / den
        cv = Fr(math.cos(th)).limit_denominator(1000)
        if abs(float(cv) - math.cos(th)) < 1e-12: coss.add(cv)
        if abs(math.cos(th)) > 1e-9:
            tv = Fr(math.tan(th)).limit_denominator(1000)
            if abs(float(tv) - math.tan(th)) < 1e-13 * max(1, abs(math.tan(th))): tans.add(tv)
print(sorted(tans), sorted(coss))

**Problem 2.15 (Ramanujan's cube roots).** Both identities numerically, and the cubic for $m=st$.

In [ ]:
cb = lambda t: math.copysign(abs(t)**(1 / 3), t)
print(sum(cb(math.cos(2 * math.pi * k / 7)) for k in (1, 2, 4)), cb((5 - 3 * cb(7)) / 2))
print(sum(cb(math.cos(2 * math.pi * k / 9)) for k in (1, 2, 4)), cb((3 * cb(9) - 6) / 2))
a, b, cc = [cb(math.cos(2 * math.pi * k / 7)) for k in (1, 2, 4)]
m = (a + b + cc) * (a * b + b * cc + cc * a)
print(m, (3 - cb(7)) / 2, 4 * m**3 - 18 * m**2 + 27 * m - 10)

**Problem 2.16 (discriminant, Zolotarev).** The discriminant of $\Phi_p$, and the sign of $x\mapsto ax$ against the Legendre
symbol.

In [ ]:
def sign(perm):
    s, seen = 1, set()
    for i in range(len(perm)):
        if i in seen: continue
        j, L = i, 0
        while j not in seen:
            seen.add(j); j = perm[j]; L += 1
        s *= (-1)**(L - 1)
    return s
ok = True
for p in sp.primerange(3, 40):
    ok &= sp.discriminant(sp.cyclotomic_poly(p, x), x) == (-1)**((p - 1) // 2) * p**(p - 2)
    ok &= all(sign([a * i % p for i in range(p)]) == sp.legendre_symbol(a, p) for a in range(1, p))
print(ok)

**Problem 2.17 (Eisenstein's sines).** The identity (a), and the products of (b) and (c), numerically.

In [ ]:
ok = True
for qq in [3, 5, 7, 9, 11, 15]:
    for xv in np.linspace(0.1, 3.0, 13):
        lhs = math.sin(qq * xv) / math.sin(xv)
        rhs = (-4)**((qq - 1) // 2) * np.prod([math.sin(xv)**2 - math.sin(2 * math.pi * k / qq)**2 for k in range(1, (qq + 1) // 2)])
        ok &= abs(lhs - rhs) < 1e-8 * max(1, abs(lhs))
for p in sp.primerange(3, 40):
    for qq in sp.primerange(3, 40):
        if p == qq: continue
        prod = np.prod([math.sin(2 * math.pi * qq * j / p) / math.sin(2 * math.pi * j / p) for j in range(1, (p + 1) // 2)])
        ok &= round(prod) == sp.legendre_symbol(qq, p)
print(ok)

**Problem 2.18 (Poisson and Jensen).** The product identity, Poisson's integral, and Jensen's formula for a random
polynomial.

In [ ]:
from scipy.integrate import quad
xv, th, n = 0.7, 0.3, 5
print(np.prod([xv**2 - 2 * xv * math.cos(th + 2 * math.pi * k / n) + 1 for k in range(n)]), xv**(2 * n) - 2 * xv**n * math.cos(n * th) + 1)
for xv in (0.5, -0.9, 2.0, -3.0):
    print(xv, quad(lambda t: math.log(1 - 2 * xv * math.cos(t) + xv * xv), 0, math.pi)[0], 2 * math.pi * math.log(max(1, abs(xv))))
co = np.random.default_rng(3).normal(size=7)
lhs = quad(lambda t: math.log(abs(np.polyval(co, cmath.exp(1j * t)))), 0, 2 * math.pi, limit=200)[0] / (2 * math.pi)
print(lhs, math.log(abs(co[0])) + sum(math.log(max(1, abs(r))) for r in np.roots(co)))

**Problem 2.19 (midpoint polygons).** The Fourier eigenvalues of $M$, and the convergence of the rescaled iterates.

In [ ]:
n = 7
w = np.exp(2j * np.pi / n)
M = (np.eye(n) + np.roll(np.eye(n), 1, axis=1)) / 2
print(np.allclose(sorted(np.abs(np.linalg.eigvals(M))), sorted(np.abs(np.cos(np.pi * np.arange(n) / n)))))
rng = np.random.default_rng(1)
z = rng.normal(size=n) + 1j * rng.normal(size=n); z -= z.mean()
zh = np.fft.fft(z) / n          # zh[j] is the coefficient of w^{jk}... with numpy's sign: z_k = sum zh[j] w^{jk}
a, b = zh[1], zh[n - 1]
for t in [5, 20, 60]:
    zt = np.linalg.matrix_power(M, t) @ z / math.cos(math.pi / n)**t
    k = np.arange(n)
    pred = a * np.exp(1j * np.pi * t / n) * w**k + b * np.exp(-1j * np.pi * t / n) * w**(-k)
    print(t, np.max(np.abs(zt - pred)))

**Problems 2.20 and 2.21 (Napoleon, Petr--Douglas--Neumann).** Napoleon's areas for a random triangle, and the theorem for
random polygons with random orders of the steps.

In [ ]:
def apex(z, th):
    e = cmath.exp(1j * th)
    return (z - e * np.roll(z, -1)) / (1 - e)
def area(z):
    return 0.5 * np.imag(np.sum(np.conj(z) * np.roll(z, -1)))
z = np.array([0, 4 + 0.5j, 1.3 + 2.2j])
P, Pi = apex(z, 2 * math.pi / 3), apex(z, -2 * math.pi / 3)
print(np.abs(P - np.roll(P, 1)), np.abs(Pi - np.roll(Pi, 1)), P.mean(), z.mean())
print(abs(area(P)) - abs(area(Pi)), area(z))
ok = True
for n in range(3, 10):
    z = rng.normal(size=n) + 1j * rng.normal(size=n)
    ks = list(range(1, n - 1)); random.shuffle(ks)
    for k in ks:
        z = apex(z, 2 * math.pi * k / n)
    ctr = z.mean()
    ok &= np.allclose((z[1:] - ctr) / (z[:-1] - ctr), cmath.exp(2j * math.pi / n))
print(ok)

**Problem 2.22 (walks on roots of unity).** Exact fourth moments by enumeration, the return probabilities for $p=3,5$
against the asymptotics, and partial sums of $\sum_NP(S_{pN}=0)$.

In [ ]:
for n in [3, 4, 5, 6]:
    w = [cmath.exp(2j * math.pi * k / n) for k in range(n)]
    for m in [1, 2, 3, 4]:
        vals = [abs(sum(s))**4 for s in itertools.product(w, repeat=m)]
        assert abs(np.mean(vals) - (2 * m * m - m)) < 1e-9
print("fourth moments ok")
ret = lambda p, N: math.exp(math.lgamma(p * N + 1) - p * math.lgamma(N + 1) - p * N * math.log(p))
for p in [3, 5]:
    for N in [10, 40, 160]:
        exact = ret(p, N)
        print(p, N, exact, math.sqrt(p) / (2 * math.pi * N)**((p - 1) / 2))
for p in [3, 5, 7]:
    s = sum(ret(p, N) for N in range(1, 400))
    print(p, "partial sum up to N=400:", s)

## Part B: laboratory

**L2.1 (cyclotomic records).** The *height* of $\Phi_n$ is its largest coefficient in absolute value. $\Phi_{105}$ is the first
of height $2$. Write a fast routine for $\Phi_n$ (use $\Phi_n=\prod_{d\mid n}(1-x^d)^{\mu(n/d)}$ with integer arrays, multiplying
by $1-x^d$ and dividing by $1-x^d$ as running sums), and find the first $n$ of height $3$, $4$ and $5$. Are they all odd? How
many distinct odd prime factors do they have?

**L2.2 (Lights Out on a grid).** On an $n\times n$ grid, pressing a cell switches it and its (up to four) neighbours. Compute
the dimension of the kernel of the pressing matrix over $\mathbb F_2$ for $n=1,\dots,20$. Check that it is $2$ for the
commercial $5\times5$ game, and find the $n$ for which every board can be solved.

**L2.3 (planar walks that escape).** Simulate the walk of Problem 2.22 for $n=3,4,5,6,8$, and count the returns to $0$ in
$10^5$ steps (use exact integer coordinates in a basis of $\Z[\zeta_n]$, not floating point). Which walks keep returning? Compare
with the prediction "recurrent iff $\varphi(n)\le2$".

**L2.4 (Lehmer's question).** For palindromic polynomials of degree $10$ with coefficients in $\{-1,0,1\}$ and leading
coefficient $1$, compute the Mahler measure $\prod\max(1,|z_k|)$ of each, and find the smallest value larger than $1$. Which
polynomial is it?

## Solutions of the laboratory

### L2.1

In [ ]:
def phi_coeffs(n):
    deg = int(sp.totient(n))
    L = n + 1   # work modulo x^(n+1); the true polynomial has degree phi(n) <= n
    a = np.zeros(L, dtype=np.int64); a[0] = 1
    for d in sp.divisors(n):
        if sp.mobius(n // d) == 1:      # multiply by 1 - x^d
            a[d:] = a[d:] - a[:L - d].copy()
    for d in sp.divisors(n):
        if sp.mobius(n // d) == -1:     # divide by 1 - x^d: running sums along each class modulo d
            pad = (-L) % d
            b = np.concatenate([a, np.zeros(pad, dtype=np.int64)]).reshape(-1, d)
            a = np.cumsum(b, axis=0).reshape(-1)[:L]
    return a[:deg + 1]
assert list(phi_coeffs(105)) == sp.Poly(sp.cyclotomic_poly(105, x), x).all_coeffs()[::-1]
records, best = [], 1
for n in range(2, 3000):
    if sp.mobius(n) == 0:       # Phi_n(x) = Phi_rad(n)(x^(n/rad n)): same height as for the radical
        continue
    h = max(abs(int(v)) for v in phi_coeffs(n))
    if h > best:
        best = h; records.append((n, h, sp.factorint(n)))
print(records)

The first heights $2,3,4,5$ appear at $105=3\cdot5\cdot7$, $385=5\cdot7\cdot11$, $1365=3\cdot5\cdot7\cdot13$ and $1785=3\cdot5\cdot7\cdot17$:
odd, with at least three prime factors, as Migotti's theorem (Problem 2.9) requires. Restricting to squarefree $n$ loses
nothing, since $\Phi_{pm}(x)=\Phi_m(x^p)$ when $p\mid m$ (Problem 2.8).

### L2.2

In [ ]:
def grid_matrix(n):
    idx = lambda i, j: i * n + j
    M = [[0] * (n * n) for _ in range(n * n)]
    for i in range(n):
        for j in range(n):
            for di, dj in ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)):
                if 0 <= i + di < n and 0 <= j + dj < n:
                    M[idx(i, j)][idx(i + di, j + dj)] = 1
    return M
dims = {n: n * n - rank_f2(grid_matrix(n)) for n in range(1, 21)}
print(dims)
print("always solvable for n =", [n for n, d in dims.items() if d == 0])

The $5\times5$ game has a kernel of dimension $2$, so exactly a quarter of the boards can be solved, as for the rings with
$3\mid n$ in Problem 2.6. The pattern of $n$ with a non-trivial kernel ($4,5,9,11,14,16,17,19,\dots$) is governed by
Chebyshev-like polynomials over $\mathbb F_2$, as Sutner showed.

### L2.3

In [ ]:
def walk_returns(n, steps, rng):
    # coordinates in the basis 1, z, ..., z^(phi(n)-1) of Z[zeta_n]: reduce z^k modulo Phi_n
    Phi = sp.Poly(sp.cyclotomic_poly(n, x), x)
    d = Phi.degree()
    vecs = []
    for k in range(n):
        r = sp.Poly(x**k, x).rem(Phi).all_coeffs()[::-1]
        vecs.append(np.array([int(v) for v in r] + [0] * (d - len(r)), dtype=np.int64))
    vecs = np.array(vecs)
    pos = np.cumsum(vecs[rng.integers(0, n, size=steps)], axis=0)
    return int(np.sum(np.all(pos == 0, axis=1)))
rng = np.random.default_rng(5)
steps, walks = 20000, 300
for n in [3, 4, 5, 6, 8]:
    mean = np.mean([walk_returns(n, steps, rng) for _ in range(walks)])
    print(n, "phi =", sp.totient(n), "mean number of returns in", steps, "steps:", mean)
lg = math.lgamma
pred3 = sum(math.exp(lg(3 * N + 1) - 3 * lg(N + 1) - 3 * N * math.log(3)) for N in range(1, steps // 3 + 1))
pred4 = sum(math.exp(2 * (lg(2 * m + 1) - 2 * lg(m + 1) - 2 * m * math.log(2))) for m in range(1, steps // 2 + 1))
pred5 = sum(math.exp(lg(5 * N + 1) - 5 * lg(N + 1) - 5 * N * math.log(5)) for N in range(1, steps // 5 + 1))
print("exact expectations: n=3", pred3, " n=4", pred4, " n=5", pred5)

The expected number of returns is $\sum_mP(S_m=0)$. For $n=3,4,6$ (rank $2$) it grows like a constant times the logarithm of
the number of steps, so it is unbounded, but slowly: about $2.5$ to $3$ returns in $20000$ steps. For $n=5$ and $n=8$ (rank
$4$) it converges, to about $0.07$ for $n=5$: most walkers never come back at all, and those that do come back early. The
simulated means agree with the exact sums.

### L2.4

In [ ]:
def mahler(co):
    return float(np.prod([max(1, abs(r)) for r in np.roots(co)]))
best = []
for half in itertools.product([-1, 0, 1], repeat=4):
    for mid in [-1, 0, 1]:
        co = [1, *half, mid, *half[::-1], 1]
        M = mahler(co)
        if M > 1 + 1e-6:
            best.append((M, co))
best.sort()
print(best[:3])
print(sp.factor(sp.Poly(best[0][1], x).as_expr()))

The winner, up to $x\mapsto-x$, is Lehmer's polynomial $x^{10}+x^9-x^7-x^6-x^5-x^4-x^3+x+1$, with Mahler measure
$1.17628\ldots$; in ninety years nobody has found a smaller value above $1$ (Problems 2.11 and 2.18).